# 🎙️ Whisper Transcription — Gradio Web App (Pause-Based Segmentation)

Ye notebook professional captioning tools (jaise Descript, foziscribe) jaisa transcription banata hai — **pauses detect karke aur punctuation ke hisab se** accurate short timestamped lines generate karta hai.

### Zaroori: Pehle GPU on karein
**Runtime → Change runtime type → Hardware accelerator → T4 GPU → Save**

## Step 1: Dependencies install karein

In [ ]:
!pip install -q openai-whisper gradio
!apt-get -qq install -y ffmpeg

## Step 2: Whisper model load karein

In [ ]:
import whisper

model = whisper.load_model("large-v3")
print("Model loaded successfully")

## Step 3: Pause-based segmentation logic

Ye function word-level timestamps leta hai aur naya segment start karta hai jab:
- Do words ke darmiyan **silence gap** ek threshold (default 0.35 sec) se zyada ho, YA
- Pichla word **punctuation** (`.` `,` `?` `!`) par khatam ho raha ho, YA
- Current line **max character limit** (default 50) tak pohanch jaye

In [ ]:
def format_mmss(seconds):
    m = int(seconds // 60)
    s = int(seconds % 60)
    return f"{m}:{s:02d}"

def srt_timestamp(seconds):
    h = int(seconds // 3600)
    m = int((seconds % 3600) // 60)
    s = int(seconds % 60)
    ms = int((seconds - int(seconds)) * 1000)
    return f"{h:02d}:{m:02d}:{s:02d},{ms:03d}"

def build_pause_based_lines(all_words, pause_threshold=0.35, max_chars=50):
    """all_words: list of dicts with 'word', 'start', 'end'"""
    lines = []
    current_words = []

    for i, w in enumerate(all_words):
        current_words.append(w)
        current_text = "".join(x["word"] for x in current_words).strip()

        is_last_word = (i == len(all_words) - 1)
        ends_with_punct = w["word"].strip().endswith((".", ",", "?", "!"))

        gap_to_next = 0
        if not is_last_word:
            gap_to_next = all_words[i + 1]["start"] - w["end"]

        should_break = (
            is_last_word
            or ends_with_punct
            or gap_to_next > pause_threshold
            or len(current_text) >= max_chars
        )

        if should_break and current_text:
            lines.append({
                "text": current_text,
                "start": current_words[0]["start"],
                "end": current_words[-1]["end"]
            })
            current_words = []

    return lines

## Step 4: Transcription function aur Gradio interface

In [ ]:
import gradio as gr
import os

def transcribe_audio(file_path, language, pause_threshold, max_chars):
    if file_path is None:
        return "Pehle koi file upload karein.", None, None

    lang = None if language == "Auto-detect" else language

    result = model.transcribe(file_path, verbose=False, language=lang, word_timestamps=True)

    # Flatten all words across all whisper segments into one list
    all_words = []
    for segment in result["segments"]:
        for w in segment.get("words", []):
            all_words.append(w)

    lines = build_pause_based_lines(all_words, pause_threshold=pause_threshold, max_chars=int(max_chars))

    # Display text, foziscribe-style: [m:ss] text
    display_lines = [f"[{format_mmss(l['start'])}] {l['text']}" for l in lines]
    display_text = f"Detected language: {result['language']}\n\n" + "\n".join(display_lines)

    base_name = os.path.splitext(os.path.basename(file_path))[0]

    txt_path = f"/content/{base_name}_transcript.txt"
    with open(txt_path, "w", encoding="utf-8") as f:
        f.write(display_text)

    srt_path = f"/content/{base_name}.srt"
    with open(srt_path, "w", encoding="utf-8") as f:
        for i, l in enumerate(lines, start=1):
            start = srt_timestamp(l["start"])
            end = srt_timestamp(l["end"])
            f.write(f"{i}\n{start} --> {end}\n{l['text']}\n\n")

    return display_text, txt_path, srt_path


languages = ["Auto-detect", "en", "ur", "hi", "ar", "fr", "es", "de", "zh", "ja"]

with gr.Blocks(title="Whisper Transcription Tool") as demo:
    gr.Markdown("# 🎙️ Audio/Video Transcription\nFile upload karein, pause-based accurate timestamped lines hasil karein.")
    with gr.Row():
        with gr.Column():
            audio_input = gr.File(label="Audio/Video file upload karein", type="filepath")
            lang_dropdown = gr.Dropdown(languages, value="Auto-detect", label="Language")
            pause_slider = gr.Slider(0.1, 1.0, value=0.2, step=0.05, label="Pause sensitivity (seconds) — kam value = zyada breaks")
            max_chars_slider = gr.Slider(20, 100, value=50, step=5, label="Max characters per line")
            submit_btn = gr.Button("Transcribe karein", variant="primary")
        with gr.Column():
            output_text = gr.Textbox(label="Transcript (pause-based timestamps)", lines=20)
            txt_file = gr.File(label="TXT file download karein")
            srt_file = gr.File(label="SRT (subtitle) file download karein")

    submit_btn.click(
        fn=transcribe_audio,
        inputs=[audio_input, lang_dropdown, pause_slider, max_chars_slider],
        outputs=[output_text, txt_file, srt_file]
    )

demo.launch(share=True, debug=True)

---
### Notes
- **Pause sensitivity** slider se control karein ke kitni chhoti pause par bhi naya segment start ho — kam value (jaise 0.2) rakhne se zyada breaks aayenge, zyada value (jaise 0.6) rakhne se kam.
- **Max characters per line** ek safety limit hai taake agar speaker continuous boley (bina rukey) to bhi line bohot lambi na ho jaye.
- Chota/tez model chahiye ho to Step 2 mein `"large-v3"` ki jagah `"medium"` likh dein.
- Timestamp format yahan `[m:ss]` hai (jaise foziscribe.ai), SRT file mein full `hh:mm:ss,ms` format hota hai jo video players ke liye standard hai.